## Data Preparation
This notebook serves as the starting point of the project, preparing the fMRI data from the open-source NSD/NSD-synthetic files by converting preprocessed beta response files into cleaner ML dataset for analysis.

#### Objectives
- Load natural and synthetic NSD files, valid voxel masks, and experimental design metadata.
- Apply valid voxel masks to reshape each beta volume into a trial * voxel matrix.
- Create binary labels for decoding (natural = 0, synthetic = 1)

#### Outputs
- X: fMRI response matrix
- y: binary stimulus labels as described above
- Processed train/test arrays for PCA
- Verification of the data structure (i.e. shapes, valid voxel counts, output dimensions) for the project report

## 1. Import Libraries & Load Data
Pre-processed fMRI data from NSD are stored in `.nii.gz` files, while experimental design metadata is stored in `.mat` files.

Here, I looked up for the dependencies for loading fMRI data since I do not have experience working with this type of dataset. Thankfully, researchers of the NSD dataset provided a comprehensive manual as well as example scripts in their lightweight GitHub repository that demonstrate usage of NSD data. Following their tutorial, I will use `nibabel` for NIfII files and `scipy.io` for MATLAB files.

To test the validity of data loading, I will only use one subject `subj01` for manipulation simplicity and expand to more subjects later in the analysis and model training files.

In [ ]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import scipy.io as sio
from tqdm import tqdm

# Define directories and file paths
base_dir = os.path.abspath(os.path.join('..', 'data', 'subj01'))
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()  # Handle case when running in interactive environment
output_dir = os.path.abspath(os.path.join(notebook_dir, '..', 'outputs', 'processed'))
os.makedirs(output_dir, exist_ok=True)

beta_files = {
    'betas_syn': os.path.join(base_dir, 'betas_nsdsynthetic.nii.gz'),
    'betas_nat_s1': os.path.join(base_dir, 'betas_session01.nii.gz'),
    'betas_nat_s2': os.path.join(base_dir, 'betas_session02.nii.gz'),
    'betas_nat_s3': os.path.join(base_dir, 'betas_session03.nii.gz')
}

mask_files = {
    'valid_syn': os.path.join(base_dir, 'valid_nsdsynthetic.nii.gz'),
    'valid_nat_s1': os.path.join(base_dir, 'valid_session01.nii.gz'),
    'valid_nat_s2': os.path.join(base_dir, 'valid_session02.nii.gz'),
    'valid_nat_s3': os.path.join(base_dir, 'valid_session03.nii.gz')
}

exp_files = {
    'exp_syn': os.path.join(base_dir, 'nsdsynthetic_expdesign.mat'),
    'exp_nat': os.path.join(base_dir, 'nsd_expdesign.mat')
}

beta_imgs = {k: nib.load(v) for k, v in beta_files.items()}
# beta_data = {k: img.get_fdata() for k, img in beta_imgs.items()} Commented out to save memory after inital attempt, see below.
mask_imgs = {k: nib.load(v) for k, v in mask_files.items()}
mask_data = {k: img.get_fdata() for k, img in mask_imgs.items()}

# To reduce memory cost, I only load the masked voxels for each trial and will stack into a trial * voxel 2D array instead of the full 4D beta file.
# If you have more RAM, feel free to comment out this section and remove comment for the beta_data line above.

# Compute shared mask
shared_mask = (
    mask_data['valid_syn'].astype(bool) &
    mask_data['valid_nat_s1'].astype(bool) &
    mask_data['valid_nat_s2'].astype(bool) &
    mask_data['valid_nat_s3'].astype(bool)
)
print(f"Shared mask valid voxel count: {shared_mask.sum()}")

# # Check if the mask and beta have the same shape
# if beta_imgs['betas_syn'].shape[:3] != shared_mask.shape:
#     raise ValueError("Shape mismatch between beta images and shared mask.")

print(beta_imgs['betas_syn'].shape[3])
print(beta_imgs['betas_nat_s1'].shape[3])


Shared mask valid voxel count: 2676916
744
750


## 2. Reshape Beta File (only one session in each condition for now)
I found out that the shapes of existing mask and beta file are not compatible. In order to apply the mask to reshape beta, I must resample the mask to the `beta_img` space. I will use `nilearn.image.resample_img` for this step.

In [ ]:
from nilearn.image import resample_img

# Resample mask to beta image space
resampled_mask_img = resample_img(
    nib.Nifti1Image(shared_mask.astype(np.uint8), mask_imgs['valid_syn'].affine),
    target_affine=beta_imgs['betas_syn'].affine,
    target_shape=beta_imgs['betas_syn'].shape[:3],
    interpolation='nearest'
)
resampled_mask = resampled_mask_img.get_fdata().astype(bool)
print(f"Resampled mask valid voxel count: {resampled_mask.sum()}")

# Save resampled mask for future use
np.save(os.path.join(output_dir, 'resampled_mask.npy'), resampled_mask)

# Extract trial-by-trial masked data for each beta file
def masked_trials(img, mask):
    n_trials = img.shape[3]
    for i in range(n_trials):
        yield img.dataobj[..., i][mask]

# Stack masked data into trial * voxel matrices
X_syn = np.stack([trial for trial in tqdm(masked_trials(beta_imgs['betas_syn'], resampled_mask))])
X_nat_s1 = np.stack([trial for trial in tqdm(masked_trials(beta_imgs['betas_nat_s1'], resampled_mask))])

# Save masked data for future use
np.save(os.path.join(output_dir, 'X_syn.npy'), X_syn)
np.save(os.path.join(output_dir, 'X_nat_s1.npy'), X_nat_s1)

print(f"X_syn shape: {X_syn.shape}")
print(f"X_nat_s1 shape: {X_nat_s1.shape}")

## 3. Prepare Data for Training
Now I shall assign binary labels `X_syn=1`and `X_nat=0` for the masked dataset. Then, I will shuffle and split the dataset for later training.

In [1]:
import numpy as np
import os

# Load masked data
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
output_dir = os.path.abspath(os.path.join(notebook_dir, '..', 'outputs', 'processed'))

X_syn = np.load(os.path.join(output_dir, 'X_syn.npy'))
X_nat_s1 = np.load(os.path.join(output_dir, 'X_nat_s1.npy'))

# Combine synthetic and natural data, create labels (1 for synthetic, 0 for natural)
X = np.concatenate([X_syn, X_nat_s1], axis=0)
y = np.concatenate([np.ones(X_syn.shape[0]), np.zeros(X_nat_s1.shape[0])])
print(f"Combined X shape: {X.shape}")
print(f"Labels shape: {y.shape}, counts: synthetic={np.sum(y==1)}, natural={np.sum(y==0)}")

# Shuffle data
n_samples = X.shape[0]
idxs = np.arange(n_samples)
np.random.seed(42)
np.random.shuffle(idxs)
X_shuffled = X[idxs]
y_shuffled = y[idxs]

# Split into train/test sets (80/20)
split_idx = int(0.8 * n_samples)
X_train, X_test = X_shuffled[:split_idx], X_shuffled[split_idx:]
y_train, y_test = y_shuffled[:split_idx], y_shuffled[split_idx:]

np.save(os.path.join(output_dir, 'X_train.npy'), X_train)
np.save(os.path.join(output_dir, 'X_test.npy'), X_test)
np.save(os.path.join(output_dir, 'y_train.npy'), y_train)
np.save(os.path.join(output_dir, 'y_test.npy'), y_test)

Combined X shape: (1494, 429603)
Labels shape: (1494,), counts: synthetic=744, natural=750


Prepared data has been saved to disk. Use `02_pca.ipynb` for dimensionality reduction next.

## Extra: Image Inspection
For an advanced part of this project (see final report for more information), I hope to test whether a generative model can distinguish natural vs synthetic images by predicting fMRI responses instead of direct image processing techniques. Since the NSD database contains all raw images for each condition, I plan to correlate them with trial indices.

In [3]:
import os
import scipy.io as sio
import numpy as np

base_dir = os.path.abspath(os.path.join('..', 'data', 'subj01'))

mat_nat = sio.loadmat(os.path.join(base_dir, 'nsd_expdesign.mat'))
mat_syn = sio.loadmat(os.path.join(base_dir, 'nsdsynthetic_expdesign.mat'))

print("Natural experiment design keys:", mat_nat.keys())
print("Synthetic experiment design keys:", mat_syn.keys())

Natural experiment design keys: dict_keys(['__header__', '__version__', '__globals__', 'basiccnt', 'masterordering', 'sharedix', 'stimpattern', 'subjectim'])
Synthetic experiment design keys: dict_keys(['__header__', '__version__', '__globals__', 'masterordering', 'stimpattern'])


#### (a) Natural Images

In [4]:
# Find image indices for natural condition
sharedix = mat_nat['sharedix'].flatten()
print(sharedix[:10])

# Find where session info is stored
for k, v in mat_nat.items():
    if isinstance(v, np.ndarray):
        print(f"{k}: shape={v.shape}")

[2951 2991 3050 3078 3147 3158 3165 3172 3182 3387]
basiccnt: shape=(3, 40)
masterordering: shape=(1, 30000)
sharedix: shape=(1, 1000)
stimpattern: shape=(40, 12, 75)
subjectim: shape=(8, 10000)


The list of `sharedix` matches their AWS database filenames. Now I can pull images using public HTTP access.

In [12]:
import requests
from tqdm import tqdm

# Get session 1 image indices from subjectim (first row)
session1_images = set(mat_nat['subjectim'][0])  # shape (10000,)

# Filter sharedix to only those in session 1
session1_mask = np.isin(mat_nat['sharedix'][0], list(session1_images))
session1_sharedix = mat_nat['sharedix'][0][session1_mask]

print(f"Session 1 valid image count: {len(session1_sharedix)}")

# Download images for session 1 valid trials
img_dir = os.path.abspath(os.path.join('..', 'img', 'nat'))
os.makedirs(img_dir, exist_ok=True)
sharedidx = mat_nat['sharedix'].flatten()

for shared_index, img_id in tqdm(zip(np.where(session1_mask)[0], session1_sharedix)):
    url = f"https://natural-scenes-dataset.s3.amazonaws.com/nsddata/stimuli/nsd/shared1000/shared{shared_index+1:04d}_nsd{img_id:05d}.png"
    out_path = os.path.join(img_dir, f"shared{shared_index+1:04d}_nsd{img_id:05d}.png")
    if not os.path.exists(out_path):
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            with open(out_path, 'wb') as f:
                f.write(r.content)
        except Exception as e:
            print(f"Failed to download {img_id}: {e}")
    else:
        pass
print("Session 1 image download complete.")


Session 1 valid image count: 1000


1000it [07:13,  2.31it/s]

Session 1 image download complete.


#### (b) Synthetic Images

In [ ]:
for k, v in mat_syn.items():
    if isinstance(v, np.ndarray):
        print(f"{k}: shape={v.shape}")

In [ ]:
import requests

# Synthetic image filenames for subject 1: nsdsynthetic221.png to nsdsynthetic284.png (64 images)
syn_base_num = 221
num_syn_images = 64
syn_img_filenames = [f"nsdsynthetic{syn_base_num + idx}.png" for idx in range(num_syn_images)]

# Download path and URL base
syn_data_dir = os.path.abspath(os.path.join('..', 'img', 'syn'))
os.makedirs(syn_data_dir, exist_ok=True)
synthetic_url_base = "https://natural-scenes-dataset.s3.amazonaws.com/nsddata/stimuli/nsdsynthetic/nsdsynthetic_subj01/"

for fname in tqdm(syn_img_filenames):
    url = synthetic_url_base + fname
    out_path = os.path.join(syn_data_dir, fname)
    if not os.path.exists(out_path):
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            with open(out_path, 'wb') as f:
                f.write(r.content)
        except Exception as e:
            print(f"Failed to download {fname}: {e}")
    else:
        pass  # Already downloaded
print("Synthetic image download complete.")
